# 02 LSTM Model

This notebook implements a Bidirectional LSTM classifier for SST-5 fine-grained sentiment classification.

The model uses:

- Tokenization and vocabulary construction
- Embedding layer
- Bidirectional LSTM
- Dropout regularization
- Fully connected classification layer


## Download and Prepare the Dataset

Run the following code to download and load the data into pandas DataFrames.

Each sample has:

- text: a short movie review snippet
- label: an integer (0-4) representing sentiment
- label_text: a human-readable version of the label

In [ ]:
import gdown
import pandas as pd

# Download SST-5 JSONL files
train_url = "https://drive.google.com/uc?id=1ERmAINKSVZ-cvnTDvgajcOlIFqu2Swdk"
test_url = "https://drive.google.com/uc?id=1Vw-SyeYij8ZspQuAzIuRizNK_If4-6iP"

train_json = "sst5_train.jsonl"
test_json = "sst5_test.jsonl"

gdown.download(train_url, train_json, quiet=False)
gdown.download(test_url, test_json, quiet=False)

# Load into pandas DataFrames
train_df = pd.read_json(train_json, lines=True)
test_df = pd.read_json(test_json, lines=True)

# Preview the data
train_df.head()

Downloading...
From: https://drive.google.com/uc?id=1ERmAINKSVZ-cvnTDvgajcOlIFqu2Swdk
To: /content/sst5_train.jsonl
100%|██████████| 1.32M/1.32M [00:00<00:00, 110MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Vw-SyeYij8ZspQuAzIuRizNK_If4-6iP
To: /content/sst5_test.jsonl
100%|██████████| 343k/343k [00:00<00:00, 83.0MB/s]


,text,label,label_text
0,"a stirring , funny and finally transporting re...",4,very positive
1,apparently reassembled from the cutting-room f...,1,negative
2,they presume their audience wo n't sit still f...,1,negative
3,the entire movie is filled with deja vu moments .,2,neutral
4,this is a visually stunning rumination on love...,3,positive


## Your Task

Your goal is to train a model (e.g., using scikit-learn, PyTorch, or transformers) that takes text as input and predicts the label (integer from 0 to 4). You can use any model architecture and preprocessing strategy you prefer.

Your final model should generate predictions on the test set. The result must be a list of integers in the following format:

`preds = [1, 0, 2, 4, 3, ...]`  

Make sure your prediction list has **the same length and order as test_df**.

In [ ]:
# Your code starts here

import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

# 1. Set hyperparameters
max_vocab_size = 20000
max_len = 100
embed_dim = 200
hidden_dim = 128
num_classes = 5
batch_size = 32
num_epochs = 8
learning_rate = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# 2. Preprocessing
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df["text"].fillna("").apply(clean_text)
test_df["clean_text"] = test_df["text"].fillna("").apply(clean_text)

# 3. Train / validation split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["clean_text"].tolist(),
    train_df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

test_texts = test_df["clean_text"].tolist()

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

# 4. Build vocabulary
counter = Counter()
for text in train_texts:
    counter.update(text.split())
vocab = {"<PAD>": 0, "<UNK>": 1}

for word, freq in counter.most_common(max_vocab_size):
    if word not in vocab:
        vocab[word] = len(vocab)
print("Vocab size:", len(vocab))

# 5. Encode tokens
def encode_text(text, vocab, max_len):
    tokens = text.split()
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    if len(ids) < max_len:
        ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return ids

X_train = [encode_text(text, vocab, max_len) for text in train_texts]
X_val = [encode_text(text, vocab, max_len) for text in val_texts]
X_test = [encode_text(text, vocab, max_len) for text in test_texts]

# 6. Define Dataset
class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = torch.tensor(texts, dtype=torch.long)
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        if self.labels is None:
            return self.texts[idx]

        return self.texts[idx], self.labels[idx]

train_dataset = TextDataset(X_train, train_labels)
val_dataset = TextDataset(X_val, val_labels)
test_dataset = TextDataset(X_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# 7. Define LSTM Model
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )


        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        hidden = self.dropout(hidden)
        out = self.fc(hidden)
        return out


model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes,
    pad_idx=vocab["<PAD>"]
)

model = model.to(device)
print(model)

# 8. Define the loss function and optimizer
label_counts = Counter(train_labels)
total_samples = len(train_labels)
class_weights = [total_samples / label_counts[i] for i in range(num_classes)]
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# 9. Define evaluation function
def evaluate(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    acc = accuracy_score(all_labels, all_preds)

    return avg_loss, macro_f1, acc

# 10. Start training
best_f1 = 0
best_model_state = None


for epoch in range(num_epochs):
    model.train()

    total_train_loss = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    val_loss, val_f1, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Macro F1: {val_f1:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")
    print("-" * 50)

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model_state = model.state_dict()

print("Best Validation Macro F1:", best_f1)

if best_model_state is not None:
    model.load_state_dict(best_model_state)


# 11. Make prediction
model.eval()
preds = []

with torch.no_grad():
    for batch_x in test_loader:
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        batch_preds = torch.argmax(outputs, dim=1)
        preds.extend(batch_preds.cpu().numpy().tolist())

print("Number of predictions:", len(preds))
print("Number of test samples:", len(test_df))
print("First 10 predictions:", preds[:10])

Using device: cpu
Train size: 6835
Validation size: 1709
Test size: 2210
Vocab size: 14710
LSTMClassifier(
  (embedding): Embedding(14710, 200, padding_idx=0)
  (lstm): LSTM(200, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=5, bias=True)
)
Epoch 1/8
Train Loss: 1.5957
Val Loss: 1.5670
Val Macro F1: 0.2493
Val Accuracy: 0.2534
--------------------------------------------------
Epoch 2/8
Train Loss: 1.4472
Val Loss: 1.4847
Val Macro F1: 0.2907
Val Accuracy: 0.2972
--------------------------------------------------
Epoch 3/8
Train Loss: 1.1798
Val Loss: 1.5766
Val Macro F1: 0.3119
Val Accuracy: 0.3288
--------------------------------------------------
Epoch 4/8
Train Loss: 0.8944
Val Loss: 1.7317
Val Macro F1: 0.3268
Val Accuracy: 0.3487
--------------------------------------------------
Epoch 5/8
Train Loss: 0.6024
Val Loss: 2.0355
Val Macro F1: 0.3375
Val Accuracy: 0.3581
------------------------------

## Evaluation

After generating your predictions, run the following code to evaluate your model.

The classification report includes precision, recall, and f1-score. The support column shows the number of true instances for each class. In addition to per-class metrics, the report also provides overall performance through accuracy, macro average (unweighted mean across classes), and weighted average (average weighted by support).

In [ ]:
from sklearn.metrics import classification_report

# True labels from test set
true_labels = test_df['label'].tolist()

# Your predictions (replace this with your actual predictions)
# preds = [...]

# Evaluation report
print(classification_report(true_labels, preds, digits=4, target_names=[
    "very negative", "negative", "neutral", "positive", "very positive"
]))


               precision    recall  f1-score   support

very negative     0.2564    0.2867    0.2707       279
     negative     0.3710    0.5972    0.4576       633
      neutral     0.2295    0.1722    0.1968       389
     positive     0.3799    0.3039    0.3377       510
very positive     0.5922    0.2657    0.3668       399

     accuracy                         0.3557      2210
    macro avg     0.3658    0.3251    0.3259      2210
 weighted avg     0.3736    0.3557    0.3440      2210

